# Phase 11 — LightGBM Lead Scoring Model

Trains a binary classifier on Retailrocket sessions to predict purchase conversion.  
Rule-based scores from Phase 10 serve as the baseline to beat.

**Label:** `converted = 1` if session contains a `transaction` event, else `0`.  
**Class imbalance:** ~0.82% conversion rate — handled via `scale_pos_weight`.  
**Metric:** ROC-AUC (primary) + Precision@K where K = top 10% of sessions.  
**Output:** `models/lead_scorer_lgbm.pkl` (joblib).

Prerequisites: `pip install -r requirements-ml.txt` and a running ClickHouse instance.

In [ ]:
import os
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import clickhouse_connect
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore', category=UserWarning)

RANDOM_STATE = 42
MODEL_PATH   = Path('../models/lead_scorer_lgbm.pkl')
MODEL_PATH.parent.mkdir(exist_ok=True)

# Feature columns — must match src/scoring/ml_scorer.py _FEATURE_COLS
FEATURE_COLS = [
    'product_views',
    'add_to_cart_count',
    'distinct_products_viewed',
    'max_scroll_pct',       # Nullable — NaN kept as-is for LightGBM
    'search_count',
    'session_duration_seconds',
]

print('LightGBM version:', lgb.__version__)

## 1. Load data from ClickHouse

In [ ]:
client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.getenv('CLICKHOUSE_PORT', '8123')),
    username=os.getenv('CLICKHOUSE_USER', 'analytics'),
    password=os.getenv('CLICKHOUSE_PASSWORD', 'analytics_password'),
    database='analytics',
)

# Use Retailrocket only — it has enough volume and the label is derivable
# from the transaction event. Live sessions have too few conversions.
result = client.query("""
    SELECT
        session_id,
        anonymous_user_id,
        product_views,
        add_to_cart_count,
        distinct_products_viewed,
        max_scroll_pct,
        search_count,
        session_duration_seconds,
        purchase_count
    FROM analytics.session_features
    WHERE source = 'retailrocket'
""")

df = pd.DataFrame(result.result_rows, columns=result.column_names)
df['max_scroll_pct'] = pd.to_numeric(df['max_scroll_pct'], errors='coerce')
df['converted'] = (df['purchase_count'] > 0).astype(int)

print(f'Sessions loaded:  {len(df):,}')
print(f'Conversions:      {df["converted"].sum():,} ({df["converted"].mean()*100:.2f}%)')
df[FEATURE_COLS].describe()

## 2. Feature matrix & train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS]
y = df['converted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Imbalance ratio — used for scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')

## 3. Train LightGBM with 5-fold stratified CV

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per fold:   {np.round(cv_scores, 4)}')

## 4. Final fit on full training set + evaluate on held-out test

In [ ]:
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_prob)
test_ap  = average_precision_score(y_test, y_prob)

# Precision@K: how many of the top-K predicted sessions are actual converters?
k = max(1, int(len(y_test) * 0.10))
top_k_idx = np.argsort(y_prob)[::-1][:k]
precision_at_k = y_test.iloc[top_k_idx].mean()

print(f'Test ROC-AUC:      {test_auc:.4f}  (target > 0.75)')
print(f'Test Avg Precision:{test_ap:.4f}')
print(f'Precision@{k}:    {precision_at_k:.4f}')

## 5. Rule-based baseline comparison

In [ ]:
import sys
sys.path.insert(0, '..')
from src.scoring.rules import SessionFeatures, score_session

test_df = X_test.copy()
test_df['converted'] = y_test.values
test_df['session_id'] = 'test'
test_df['anonymous_user_id'] = 'test'
test_df['source'] = 'retailrocket'
# Retailrocket sessions always have these zero-value fields
for col in ('page_views', 'search_count', 'purchase_count', 'cart_abandoned'):
    test_df[col] = 0
test_df['add_to_cart_count'] = test_df['add_to_cart_count'].astype(int)

rule_scores = []
for _, row in test_df.iterrows():
    sf = SessionFeatures(
        session_id=row['session_id'],
        anonymous_user_id=row['anonymous_user_id'],
        page_views=int(row['page_views']),
        product_views=int(row['product_views']),
        add_to_cart_count=int(row['add_to_cart_count']),
        purchase_count=int(row['purchase_count']),
        search_count=int(row['search_count']),
        max_scroll_pct=None,
        session_duration_seconds=int(row['session_duration_seconds']),
        distinct_products_viewed=int(row['distinct_products_viewed']),
        cart_abandoned=int(row['cart_abandoned']),
        source=row['source'],
    )
    rule_scores.append(score_session(sf).lead_score)

rule_auc = roc_auc_score(y_test, rule_scores)
print(f'Rule-based ROC-AUC: {rule_auc:.4f}')
print(f'LightGBM ROC-AUC:   {test_auc:.4f}  (delta: {test_auc - rule_auc:+.4f})')

## 6. Feature importances

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=True)

importance.plot.barh(figsize=(7, 4), title='Feature Importances (gain)')
plt.tight_layout()
plt.show()
print(importance.sort_values(ascending=False))

## 7. Save model

In [ ]:
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')
print(f'\nModel card:')
print(f'  Algorithm:        LightGBM binary classifier')
print(f'  Training rows:    {len(X_train):,}')
print(f'  CV ROC-AUC:       {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Test ROC-AUC:     {test_auc:.4f}')
print(f'  Precision@10%:    {precision_at_k:.4f}')
print(f'  Features:         {FEATURE_COLS}')
print(f'  Known gaps:       max_scroll_pct always NaN for Retailrocket')
print(f'                    search_count always 0 for Retailrocket')